<a href="https://colab.research.google.com/github/rumali-DS/S-P-500-Stock-Segmentation-RFM/blob/main/task1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 6BUIS017W CWK1 - Task 1: Data preparation and feature engineering




### CELL 1 - Setup, versions, constants

In [ ]:
!pip install -q yfinance lxml html5lib
import sys, io, os, datetime as dt
import json
from pathlib import Path
import numpy as np
import pandas as pd
import requests
import yfinance as yf

print("Python", sys.version.split()[0], "| pandas", pd.__version__,
      "| numpy", np.__version__, "| yfinance", yf.__version__)

OUT = "data"
os.makedirs(OUT, exist_ok=True)

START, END = "2021-01-01", "2025-01-01"   # end date is exclusive
BENCH = "^GSPC"
WIKI_URL = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"

# Use the existing saved CSVs only if they are from the same study period and ticker list.
USE_CACHE = False  # Change to True to reuse your saved data/ folder.
CACHE_CONSTITUENTS = "sp500_constituents_2026-11-18.csv"  # Verify filename.
CACHE_DOWNLOAD_DATE = None  # Enter actual original date if no acquisition metadata exists.
EXCLUDE_ZERO_VOLUME = True  # Conservative completeness rule; review zero-volume evidence.

Python 3.13.18 | pandas 2.2.3 | numpy 2.4.6 | yfinance 0.2.66


### CELL 2 - 1.1 Ticker list from Wikipedia (record the retrieval date)

In [ ]:
if USE_CACHE:
    cons = pd.read_csv(f"{OUT}/{CACHE_CONSTITUENTS}")
    assert cons["ticker"].is_unique
    assert cons["retrieval_date"].nunique() == 1
    RETRIEVAL_DATE = str(cons["retrieval_date"].iloc[0])
    print(len(cons), "cached tickers; originally retrieved on", RETRIEVAL_DATE)
else:
    RETRIEVAL_DATE = dt.date.today().isoformat()
    response = requests.get(WIKI_URL, headers={"User-Agent": "Mozilla/5.0 (student coursework)"},
                            timeout=30)
    response.raise_for_status()
    html = response.text
    cons = pd.read_html(io.StringIO(html))[0]
    cons = cons.rename(columns={"Symbol": "ticker_wiki", "Security": "name",
                                "GICS Sector": "sector", "GICS Sub-Industry": "sub_industry"})
    # Yahoo uses '-' where Wikipedia uses '.' (e.g. BRK.B -> BRK-B)
    cons["ticker"] = cons["ticker_wiki"].str.replace(".", "-", regex=False)
    cons["retrieval_date"] = RETRIEVAL_DATE
    cons = cons[["ticker_wiki", "ticker", "name", "sector", "sub_industry", "retrieval_date"]]
    assert cons["ticker"].is_unique
    cons.to_csv(f"{OUT}/sp500_constituents_{RETRIEVAL_DATE}.csv", index=False)
    print(len(cons), "tickers retrieved on", RETRIEVAL_DATE)

503 tickers retrieved on 2026-11-23


### CELL 3 - 1.1 Download daily data (yfinance), save a snapshot

In [ ]:
FIELDS = ["Adj Close", "Close", "Volume", "Dividends", "Stock Splits"]
meta_path = Path(OUT) / "acquisition_metadata.json"
if USE_CACHE:
    raw = {f: pd.read_csv(f"{OUT}/raw_{f.replace(' ', '_').lower()}.csv",
                          index_col=0, parse_dates=True)
           for f in FIELDS if Path(f"{OUT}/raw_{f.replace(' ', '_').lower()}.csv").exists()}
    if meta_path.exists():
        meta = json.loads(meta_path.read_text())
        assert (meta["start"], meta["end"]) == (START, END)
        assert meta["retrieval_date"] == RETRIEVAL_DATE
        assert set(meta["tickers"]) == set(cons["ticker"]) | {BENCH}
        DOWNLOAD_DATE = meta["download_date"]
    else:
        DOWNLOAD_DATE = CACHE_DOWNLOAD_DATE
        if DOWNLOAD_DATE is None:
            print("Original download date unrecorded; verify and enter CACHE_DOWNLOAD_DATE in Cell 1.")
else:
    # Request separate Close and Adj Close fields and corporate action records.
    # Verify adjustment conventions rather than assuming they are identical.
    tickers = cons["ticker"].tolist() + [BENCH]
    frames = {}
    for i in range(0, len(tickers), 100):
        d = yf.download(tickers[i:i + 100], start=START, end=END, auto_adjust=False,
                        actions=True, progress=False, threads=True, group_by="column")
        for field in ["Adj Close", "Close", "Volume", "Dividends", "Stock Splits"]:
            if field in d.columns.get_level_values(0):
                frames.setdefault(field, []).append(d[field])

    raw = {f: pd.concat(v, axis=1) for f, v in frames.items()}
    for f, df in raw.items():
        df.index = pd.to_datetime(df.index).tz_localize(None)
        assert not df.columns.duplicated().any(), f"Duplicate ticker columns: {f}"
        df = df.sort_index()
        raw[f] = df
    DOWNLOAD_DATE = dt.date.today().isoformat()

# Check the snapshot before saving or proceeding; no silent price fallback.
assert all(f in raw for f in ["Adj Close", "Close", "Volume"]), "Missing required price/volume field"
for f in ["Adj Close", "Close", "Volume"]:
    assert BENCH in raw[f].columns, f"Benchmark missing from {f}"
    assert raw[f].index.is_unique, f"Duplicate dates in {f}"
    assert raw[f].index.min() >= pd.Timestamp(START) and raw[f].index.max() < pd.Timestamp(END)
assert raw["Adj Close"][BENCH].notna().any(), "Benchmark download failed"
if not USE_CACHE:
    for f, df in raw.items():
        df.to_csv(f"{OUT}/raw_{f.replace(' ', '_').lower()}.csv")
    meta_path.write_text(json.dumps({"start": START, "end": END,
        "retrieval_date": RETRIEVAL_DATE, "download_date": DOWNLOAD_DATE,
        "tickers": tickers, "source": "Yahoo Finance via yfinance", "yfinance": yf.__version__}, indent=2))
print({f: df.shape for f, df in raw.items()})
print("Date range:", raw["Close"].index.min().date(), "->", raw["Close"].index.max().date())


ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['FDXF']: YFPricesMissingError('possibly delisted; no price data found  (1d 2021-01-01 -> 2025-01-01) (Yahoo error = "Data doesn\'t exist for startDate = 1609477200, endDate = 1735707600")')
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['HONA']: YFPricesMissingError('possibly delisted; no price data found  (1d 2021-01-01 -> 2025-01-01) (Yahoo error = "Data doesn\'t exist for startDate = 1609477200, endDate = 1735707600")')
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['Q']: YFPricesMissingError('possibly delisted; no price data found  (1d 2021-01-01 -> 2025-01-01) (Yahoo error = "Data doesn\'t exist for startDate = 1609477200, endDate = 1735707600")')
ERROR:yfinance:
2 Failed downloads:
ERROR:yfinance:['SNDK', 'VYLR']: YFPricesMissingError('possibly delisted; no price data found  (1d 2021-01-01 -> 2025-01-01) (Yahoo error = "Data doesn\'t exist for startDate = 1609477200, endDate = 1735707600")')


{'Adj Close': (1005, 504), 'Close': (1005, 504), 'Volume': (1005, 504), 'Dividends': (1005, 499), 'Stock Splits': (1005, 499)}
Date range: 2021-01-04 -> 2024-12-31


### CELL 4 - 1.2 Validation and alignment

In [ ]:
adj, close, vol = raw["Adj Close"], raw["Close"], raw["Volume"]
splits, divs = raw.get("Stock Splits"), raw.get("Dividends")

# Check the benchmark against the union of recorded dates before reindexing.
all_dates = adj.index.union(close.index).union(vol.index).sort_values()
for label, frame in [("Adj Close", adj), ("Close", close), ("Volume", vol)]:
    duplicates = int(frame.index.duplicated().sum())
    print(label, "duplicate dates:", duplicates)
    assert duplicates == 0, f"Resolve duplicate dates in {label} first"
for label, frame in [("Adj Close", adj), ("Close", close)]:
    b = frame[BENCH].reindex(all_dates)
    print(label, "benchmark missing:", int(b.isna().sum()),
          "nonpositive:", int((b <= 0).sum()))
    assert np.isfinite(b).all() and (b > 0).all(), f"Invalid benchmark {label}"
cal = adj[BENCH].dropna().index.sort_values()
extra_dates = all_dates.difference(cal)
print("Dates not in benchmark calendar:", len(extra_dates))
# Expected US equity sessions for this fixed study period (compare with downloaded dates).
# This count and endpoint check cannot detect every possible calendar mismatch.
assert len(cal) == 1005 and cal[0] == pd.Timestamp("2021-01-04") and cal[-1] == pd.Timestamp("2024-12-31"), "Unexpected benchmark coverage: inspect before continuing"
adj, close, vol = (x.reindex(cal) for x in (adj, close, vol))
splits = splits.reindex(cal) if splits is not None else None
divs = divs.reindex(cal) if divs is not None else None

rows = []
for t in cons["ticker"]:
    r = {"ticker": t, "reasons": []}
    if t not in adj.columns or adj[t].isna().all():
        r["reasons"].append("no adjusted price data returned for study period")
        rows.append(r); continue
    if t not in close.columns or t not in vol.columns:
        r["reasons"].append("missing Close or Volume field")
        rows.append(r); continue
    r["n_missing_adj"] = int(adj[t].isna().sum())
    r["n_missing_close"] = int(close[t].isna().sum())
    r["n_missing_vol"] = int(vol[t].isna().sum())
    r["first_valid_date"] = adj[t].first_valid_index()
    r["n_nonpositive_price"] = int(((adj[t] <= 0) | (close[t] <= 0)).sum())
    r["n_negative_vol"] = int((vol[t] < 0).sum())
    r["n_zero_vol"] = int((vol[t] == 0).sum())
    r["n_infinite"] = int(np.isinf(adj[t]).sum() + np.isinf(close[t]).sum() + np.isinf(vol[t]).sum())
    r["n_splits"] = int((splits[t] > 0).sum()) if splits is not None and t in splits else np.nan
    r["n_dividends"] = int((divs[t] > 0).sum()) if divs is not None and t in divs else np.nan
    dret = adj[t].pct_change(fill_method=None)
    r["max_abs_daily_ret"] = float(dret.abs().max())
    r["jump_flag"] = bool(r["max_abs_daily_ret"] > 0.40)           # check vs recorded splits
    if r["n_missing_adj"] or r["n_missing_close"] or r["n_missing_vol"]:
        r["reasons"].append("missing values / unequal history")
    if r["n_nonpositive_price"]:
        r["reasons"].append("non-positive price")
    if r["n_negative_vol"]:
        r["reasons"].append("negative volume")
    if r["n_infinite"]:
        r["reasons"].append("nonfinite price or volume")
    if EXCLUDE_ZERO_VOLUME and r["n_zero_vol"] > 0:
        r["reasons"].append("zero-volume sessions (conservative completeness rule)")
    rows.append(r)

val = pd.DataFrame(rows)
val["excluded"] = val["reasons"].str.len() > 0
val["reasons"] = val["reasons"].apply("; ".join)
val.to_csv(f"{OUT}/validation_log.csv", index=False)

valid = val.loc[~val["excluded"], "ticker"].tolist()
print("Retained:", len(valid), "| Excluded:", int(val["excluded"].sum()))
print(val.loc[val["excluded"], ["ticker", "reasons"]].to_string(index=False))
print("Jump flags (inspect vs splits):")
print(val.loc[val["jump_flag"] == True, ["ticker", "max_abs_daily_ret", "n_splits"]])

# Show actual zero-volume observations so the exclusion policy can be reviewed.
zero_rows = []
for t in val.loc[val["n_zero_vol"].fillna(0) > 0, "ticker"]:
    for day in vol.index[vol[t].eq(0)]:
        zero_rows.append({"ticker": t, "date": day, "Close": close.at[day, t],
                          "Adj Close": adj.at[day, t], "Volume": vol.at[day, t]})
zero_log = pd.DataFrame(zero_rows, columns=["ticker", "date", "Close", "Adj Close", "Volume"])
zero_log.to_csv(f"{OUT}/zero_volume_review.csv", index=False)
print("Zero-volume evidence:", zero_log.head(20).to_string(index=False))
print("Validation totals (available records):")
print(val[["n_missing_adj", "n_missing_close", "n_missing_vol", "n_nonpositive_price",
           "n_negative_vol", "n_zero_vol", "n_infinite", "n_splits", "n_dividends"]].sum(min_count=1))
assert valid, "No shares passed validation"

# Corporate action diagnostics: compare dates instead of inferring causation from counts.
ratio_changes = (adj[valid] / close[valid]).pct_change(fill_method=None)
ratio_jump = ratio_changes.abs().max().nlargest(5)
print("Largest single-day absolute changes in Adj Close / Close:")
print(ratio_jump)

def action_value(frame, t, day):
    return frame.at[day, t] if frame is not None and t in frame.columns else np.nan

review = []
for t in valid:
    ret = adj[t].pct_change(fill_method=None)
    days = set(ret.index[ret.abs() > 0.40])  # chosen diagnostic, not an exclusion rule
    if t in ratio_jump.index and ratio_changes[t].notna().any():
        days.add(ratio_changes[t].abs().idxmax())
    if splits is not None and t in splits.columns:
        days.update(splits.index[splits[t] > 0])
    for day in sorted(days):
        prev_pos = cal.get_loc(day) - 1
        prev = cal[prev_pos] if prev_pos >= 0 else None
        review.append({"ticker": t, "date": day, "adjusted_daily_return": ret.at[day],
                       "ratio_change": ratio_changes.at[day, t],
                       "split": action_value(splits, t, day),
                       "dividend": action_value(divs, t, day),
                       "previous_close": close.at[prev, t] if prev is not None else np.nan,
                       "close": close.at[day, t],
                       "previous_volume": vol.at[prev, t] if prev is not None else np.nan,
                       "volume": vol.at[day, t]})
review_cols = ["ticker", "date", "adjusted_daily_return", "ratio_change", "split", "dividend",
               "previous_close", "close", "previous_volume", "volume"]
action_review = pd.DataFrame(review, columns=review_cols)
action_review.to_csv(f"{OUT}/corporate_action_review.csv", index=False)
print("Extreme return and ratio-change dates (retained shares):")
print(action_review.loc[(action_review["adjusted_daily_return"].abs() > 0.40) |
                        action_review["ticker"].isin(ratio_jump.index)].to_string(index=False))
# Review these flags against provider records; absence of an action is not proof of its cause.


Adj Close duplicate dates: 0
Close duplicate dates: 0
Volume duplicate dates: 0
Adj Close benchmark missing: 0 nonpositive: 0
Close benchmark missing: 0 nonpositive: 0
Dates not in benchmark calendar: 0
Retained: 486 | Excluded: 17
ticker                                               reasons
   APP                      missing values / unequal history
  COIN                      missing values / unequal history
   CEG                      missing values / unequal history
   EXE                      missing values / unequal history
  FDXF      no adjusted price data returned for study period
  FERG zero-volume sessions (conservative completeness rule)
  GEHC                      missing values / unequal history
   GEV                      missing values / unequal history
  HONA      no adjusted price data returned for study period
     Q      no adjusted price data returned for study period
  RDDT                      missing values / unequal history
  HOOD                      missing 

### CELL 5 - 1.3 Weekly returns (Friday-ending) and weekly beta

In [ ]:
cols = valid + [BENCH]
wk = adj[cols].resample("W-FRI").last()          # last available trading price in each week

# Drop incomplete first / final calendar weeks (week start = Monday, week end = Friday)
first_day, last_day = pd.Timestamp(START), pd.Timestamp(END) - pd.Timedelta(days=1)
week_end = wk.index
week_start = week_end - pd.Timedelta(days=4)
wk = wk[(week_start >= first_day) & (week_end <= last_day)]

# Alignment checks: no missing weeks, no missing prices, identical labels for all shares + benchmark
assert wk.isna().sum().sum() == 0, "Missing weekly prices"
assert (wk.index.to_series().diff().dropna() == pd.Timedelta(days=7)).all(), "Gap in week labels"
print("Weekly price points:", len(wk), "|", wk.index.min().date(), "->", wk.index.max().date())

wk_ret = wk.pct_change(fill_method=None).iloc[1:]   # r_t = P_t / P_{t-1} - 1
assert np.isfinite(wk_ret.to_numpy()).all(), "Invalid weekly returns"
N_WEEKS = len(wk_ret)
assert N_WEEKS > 1 and wk_ret[BENCH].var() > 0, "Invalid benchmark variance"
print("Aligned weekly return observations:", N_WEEKS)

mkt = wk_ret[BENCH]
mean_ret = wk_ret[valid].mean()                                   # arithmetic mean
beta = wk_ret[valid].apply(lambda s: s.cov(mkt)) / mkt.var()      # cov / var (both ddof=1)

# Independent check of beta (OLS slope) on a few shares
for t in valid[:3]:
    slope = np.polyfit(mkt.values, wk_ret[t].values, 1)[0]
    assert np.isclose(slope, beta[t]), t
print("Beta check vs OLS slope: OK | benchmark mean weekly return:", round(mkt.mean(), 5))
wk_ret.to_csv(f"{OUT}/weekly_returns.csv")

Weekly price points: 208 | 2021-01-08 -> 2024-12-27
Aligned weekly return observations: 207
Beta check vs OLS slope: OK | benchmark mean weekly return: 0.0024


In [ ]:
weekly_summary = pd.DataFrame({
    "Mean Weekly Return (%)": mean_ret * 100,
    "Weekly Beta": beta
})

display(weekly_summary.head(10).round(4))

,Mean Weekly Return (%),Weekly Beta
Ticker,,
MMM,0.1142,0.8691
AOS,0.1991,1.0428
ABT,0.0971,0.5835
ABBV,0.3749,0.4254
ACN,0.2376,1.1433
ADBE,0.0990,1.5535
AMD,0.3624,1.8776
AES,-0.1420,1.1439
AFL,0.5004,0.7072


### CELL 6 - 1.4 Adapted RFM (observation date, rolling threshold, R, F, M)

In [ ]:
OBS_DATE = cal[-1]                                   # final common completed trading date
assert adj.loc[OBS_DATE, valid].notna().all() and vol.loc[OBS_DATE, valid].notna().all()
assert len(cal) >= 80, "Need >= 20 baseline + 60 window sessions"
print("Observation date:", OBS_DATE.date())

V = vol[valid]
C = close[valid]  # Use supplied Close; review price/volume split compatibility before interpreting M.

# Baseline = mean volume of the PRECEDING 20 sessions, excluding the day tested (shift(1))
baseline = V.shift(1).rolling(20, min_periods=20).mean()
threshold = 2 * baseline
flag = V > threshold  # strictly greater than 2x; unavailable baseline -> False
assert threshold.tail(60).notna().all().all(), "Insufficient baseline in final 60 sessions"

# Recency: latest event in the whole valid history, in CALENDAR days
last_event = flag.apply(lambda s: s[s].index.max() if s.any() else pd.NaT)
R = (OBS_DATE - pd.to_datetime(last_event)).dt.days          # NaN if no event (NOT zero)

# Frequency: count of event days in the latest 60 sessions
F = flag.tail(60).sum()

# Monetary: sum of Close x Volume over the latest 60 sessions (USD)
M = (C * V).tail(60).sum(min_count=60)
assert F.between(0, 60).all() and R.dropna().ge(0).all()
assert np.isfinite(M).all() and M.ge(0).all()

# --- Verification: brute-force loop for one share must match the vectorised flags
t0 = valid[0]
v0 = V[t0].values
manual = np.zeros(len(v0), dtype=bool)
for i in range(20, len(v0)):
    manual[i] = v0[i] > 2 * v0[i - 20:i].mean()      # i-20..i-1 only: no look-ahead
assert (manual == flag[t0].values).all()
print("Threshold alignment / no look-ahead check passed for", t0)

Observation date: 2024-12-31
Threshold alignment / no look-ahead check passed for MMM


### CELL 7 - Worked example

In [ ]:
EX = "AAPL" if "AAPL" in valid else valid[0]
we = pd.DataFrame({"Close": C[EX], "Volume": V[EX], "Baseline20": baseline[EX],
                   "Threshold": threshold[EX], "HighVol": flag[EX],
                   "TradedValue": C[EX] * V[EX]}).tail(60)
we.to_csv(f"{OUT}/worked_example_{EX}.csv")
print("Qualifying events in final 60 sessions:")
print(we.loc[we["HighVol"]])
print("60-session traded value:", we["TradedValue"].sum())
print(f"{EX}: last event = {pd.to_datetime(last_event[EX]).date()}, R = {R[EX]} days, "
      f"F = {int(F[EX])}, M = ${M[EX]:,.0f}")

Qualifying events in final 60 sessions:
                 Close     Volume  Baseline20   Threshold  HighVol  \
Date                                                                 
2024-11-25  232.869995   90152800  43820740.0  87641480.0     True   
2024-12-20  254.490005  147495300  45004605.0  90009210.0     True   

             TradedValue  
Date                      
2024-11-25  2.099388e+10  
2024-12-20  3.753608e+10  
60-session traded value: 623303377706.5842
AAPL: last event = 2024-12-20, R = 11 days, F = 2, M = $623,303,377,707


### CELL 8 - Feature table, exclusion log, summaries

In [ ]:
feat = pd.DataFrame({"mean_weekly_return": mean_ret, "weekly_beta": beta,
                     "n_weekly_obs": N_WEEKS, "last_event_date": last_event,
                     "R": R, "F": F, "M": M})
feat = feat.join(cons.set_index("ticker")[["name", "sector"]])
feat.index.name = "ticker"
feat["rfm_missing"] = feat["R"].isna()
num = ["mean_weekly_return", "weekly_beta", "R", "F", "M"]
feat["nonfinite_features"] = ~np.isfinite(feat[num]).all(axis=1)

print("Shares with no high-volume event (R missing):", int(feat["rfm_missing"].sum()))
common = feat[~feat["nonfinite_features"]].copy()          # COMMON valid set used for ALL clustering
print("Common valid share set:", len(common))

excl = val.loc[val["excluded"], ["ticker", "reasons"]].copy()
no_event = feat.loc[feat["rfm_missing"]].reset_index()
no_event = no_event[["ticker"]].assign(reasons="no high-volume event (R missing)")
other_invalid = feat.loc[feat["nonfinite_features"] & ~feat["rfm_missing"]].reset_index()[["ticker"]].assign(reasons="other nonfinite feature")
pd.concat([excl, no_event, other_invalid], ignore_index=True).to_csv(f"{OUT}/exclusion_log.csv", index=False)

feat.to_csv(f"{OUT}/feature_table_all_valid.csv")
common.to_csv(f"{OUT}/feature_table_common.csv")

num = ["mean_weekly_return", "weekly_beta", "R", "F", "M"]
print(common[num].describe().T)
print("Skewness:\n", common[num].skew())
print("Log1p(M) exploratory summary:\n", np.log1p(common["M"]).describe())

# Provenance note for the report / README
with open(f"{OUT}/run_metadata.txt", "w") as f:
    f.write(f"Wikipedia retrieval date: {RETRIEVAL_DATE}\nOriginal download date: {DOWNLOAD_DATE}\nCached data used: {USE_CACHE}\nExclude zero volume: {EXCLUDE_ZERO_VOLUME}\nStudy period: {START} to {END} (end exclusive)\n"
            f"Observation date: {OBS_DATE.date()}\nWeekly observations: {N_WEEKS}\n"
            f"Shares requested: {len(cons)} | retained after validation: {len(valid)} | "
            f"common RFM set: {len(common)}\n"
            f"Packages: python {sys.version.split()[0]}, pandas {pd.__version__}, "
            f"numpy {np.__version__}, yfinance {yf.__version__}\n")

Shares with no high-volume event (R missing): 0
Common valid share set: 486
                    count          mean           std           min  \
mean_weekly_return  486.0  2.857173e-03  2.651609e-03 -4.468093e-03   
weekly_beta         486.0  9.736953e-01  4.073389e-01  5.545250e-02   
R                   486.0  2.040741e+01  2.134244e+01  4.000000e+00   
F                   486.0  2.989712e+00  1.330800e+00  0.000000e+00   
M                   486.0  3.940419e+10  1.291388e+11  1.297483e+09   

                             25%           50%           75%           max  
mean_weekly_return  1.181530e-03  2.618933e-03  4.207323e-03  1.777248e-02  
weekly_beta         6.665995e-01  9.506096e-01  1.179538e+00  3.778560e+00  
R                   1.100000e+01  1.100000e+01  1.475000e+01  2.220000e+02  
F                   2.000000e+00  3.000000e+00  4.000000e+00  8.000000e+00  
M                   1.019517e+10  1.668382e+10  3.227689e+10  1.847697e+12  
Skewness:
 mean_weekly_return     0